# 07. Evaluación de Sistemas RAG
**Objetivo:** Implementar un sistema de evaluación objetiva para el `Steam-Support-Bot`. En este módulo, mediremos la calidad de las respuestas para optimizar el comportamiento del bot y mitigar alucinaciones.


### Paso 0: Preparación del Entorno y Librerías

Para realizar la evaluación de nuestro sistema RAG, necesitamos herramientas de análisis y visualización de datos, además del motor de LangChain.

**Librerías principales utilizadas en este módulo:**
* `langchain`, `langchain-openai`, `langchain-community`: Para orquestar los LLMs y la base de datos vectorial.
* `faiss-cpu`: Motor de búsqueda vectorial de alto rendimiento (creado por Meta) que usaremos de forma local.
* `pandas`: Para estructurar nuestros resultados de evaluación en un reporte tabular (DataFrames).
* `python-dotenv`: Para manejar de forma segura nuestras llaves (Tokens de GitHub).


# Instalación de dependencias para evaluación

pip install -qU langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu pandas python-dotenv langsmith

(Si no dice nada, es que se instalo correctamente)

### Importaciones y Conexión de Motores
Aquí levantamos a "los jueces" (los modelos de IA que van a evaluar a nuestro bot) usando las mismas credenciales de GitHub que usó tu compañero.

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv

# Componentes de LangChain
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

# Cargar variables de entorno
load_dotenv()
github_token = os.getenv("GITHUB_TOKEN")
github_base_url = os.getenv("GITHUB_BASE_URL", "https://models.inference.ai.azure.com")

if not github_token:
    raise EnvironmentError("❌ GITHUB_TOKEN no encontrado. Revisa tu archivo .env")

print("✅ Credenciales cargadas correctamente.")

# 2. Configurar los motores (Usaremos GPT-4o como juez que evaluara)
juez_llm = ChatOpenAI(
    base_url=github_base_url,
    api_key=github_token,
    model="gpt-4o",
    temperature=0.0 # Temperatura 0 para que la evaluación sea estricta
)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=github_token,
    base_url=github_base_url
)

print("⚖️ Motor LLM Evaluador (Juez) y Embeddings inicializados.")

✅ Credenciales cargadas correctamente.
⚖️ Motor LLM Evaluador (Juez) y Embeddings inicializados.


In [ ]:
# Extraemos el conocimiento exacto que usó el bot original
steam_knowledge = """POLÍTICAS DE REEMBOLSO: 
- Juegos: Menos de 2 horas de juego y menos de 14 días desde la compra.
- DLC: Reembolsable si el juego base no se ha jugado más de 2 horas tras la compra del DLC.
- Preventas: Se puede pedir el reembolso en cualquier momento antes del lanzamiento.

REQUISITOS DE JUEGOS:
- ELDEN RING: Mínimo 12GB RAM, i5-8400, GTX 1060 (6GB). Recomendado 16GB RAM y RTX 3060.
- CYBERPUNK 2077: Mínimo 12GB RAM, GTX 1060 (6GB) y OBLIGATORIO instalar en SSD.
- COUNTER-STRIKE 2 (CS2): Mínimo 8GB RAM, procesador de 4 hilos, dedicada con 1GB compatible con Shader Model 5.0.

SOPORTE TÉCNICO Y ERRORES:
- Error 'Disco Corrupto' o 'Error de escritura': Se soluciona borrando la caché de descargas en Parámetros > Descargas.
- Juego no inicia: Verificar integridad de archivos o reinstalar DirectX y Visual C++ Redistributable.

SEGURIDAD Y CUENTAS:
- Steam Guard (2FA): Sistema de seguridad obligatorio para habilitar intercambios inmediatos en el Mercado de la Comunidad.
- Cuentas Bloqueadas: Puede ocurrir por 'Chargeback' (devolución de cargo bancaria), intentos de inicio fallidos masivos o uso de VPN para comprar en otras regiones más baratas."""

# Recreamos el motor de búsqueda (Retriever)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = text_splitter.split_text(steam_knowledge)

# Creamos la base de datos en memoria
vector_db = FAISS.from_texts(texts=chunks, embedding=embeddings)
retriever = vector_db.as_retriever(search_kwargs={"k": 2})

print(f"📦 Base de datos de evaluación lista con {len(chunks)} fragmentos.")

📦 Base de datos de evaluación lista con 6 fragmentos.


Vamos a crear 3 preguntas. En la cual la pregunta 2 es una trampa a propósito (Lethal Company no está en el texto de arriba), para ver si el bot inventa cosas o si es honesto.

In [3]:
dataset_evaluacion = [
    {
        "id": 1,
        "pregunta": "Compré el Cyberpunk 2077 hace 5 días y jugué 1 hora. ¿Puedo pedir reembolso?",
        "respuesta_esperada": "Sí, cumple con la regla de menos de 14 días y menos de 2 horas de juego."
    },
    {
        "id": 2,
        "pregunta": "¿Cuánta RAM necesito para jugar Lethal Company?",
        "respuesta_esperada": "No hay información en el contexto sobre Lethal Company." # Pregunta trampa
    },
    {
        "id": 3,
        "pregunta": "Mi cuenta fue bloqueada, ¿por qué pudo haber sido?",
        "respuesta_esperada": "Puede ocurrir por Chargeback, intentos de inicio fallidos o uso de VPN."
    }
]

print(f"🎯 Dataset creado con {len(dataset_evaluacion)} casos de prueba.")

🎯 Dataset creado con 3 casos de prueba.


### El Motor RAG y el Evaluador

Aquí conectamos el buscador con el generador para simular las respuestas y creamos la función que califica la Fidelidad (Faithfulness).

In [ ]:
import json
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# ARMAMOS EL BOT PARA RESPONDER
prompt_rag = ChatPromptTemplate.from_messages([
    ("system", "Eres soporte de Steam. Responde SOLO basándote en este contexto:\n\n{context}\n\nSi la información no está en el contexto, di: 'No tengo información sobre eso'."),
    ("human", "{input}")
])

cadena_qa = create_stuff_documents_chain(juez_llm, prompt_rag)
cadena_rag = create_retrieval_chain(retriever, cadena_qa)

# 2. ARMAMOS EL JUEZ PARA EVALUAR
def evaluar_fidelidad(pregunta, contexto, respuesta_bot):
    prompt_juez = f"""
    Eres un auditor estricto. Revisa si la RESPUESTA DEL BOT inventa información que no está en el CONTEXTO.
    Si el bot dice que no tiene información (y es cierto), eso es ALTA FIDELIDAD (10).
    
    PREGUNTA: {pregunta}
    CONTEXTO: {contexto}
    RESPUESTA DEL BOT: {respuesta_bot}
    
    Evalúa del 0 al 10. Responde ÚNICAMENTE en JSON con este formato exacto:
    {{"puntaje": 10, "razon": "breve motivo"}}
    """
    
    try:
        resultado = juez_llm.invoke(prompt_juez)
        texto = resultado.content.replace("```json", "").replace("```", "").strip()
        return json.loads(texto)
    except Exception as e:
        return {"puntaje": 0, "razon": "Error al evaluar"}

print("⚙️ Motor RAG y Función de Evaluación (Fidelidad) listos.")

⚙️ Motor RAG y Función de Evaluación (Fidelidad) listos.


### Ejecución y Tabla de Pandas


In [ ]:
resultados_finales = []

print("🚀 Iniciando auditoría automática...\n")

for caso in dataset_evaluacion:
    print(f"Consultando: '{caso['pregunta']}'...")
    
    # El bot responde
    respuesta_rag = cadena_rag.invoke({"input": caso['pregunta']})
    respuesta_bot = respuesta_rag['answer']
    
    # Extraemos el contexto que usó el bot
    contexto_usado = "\n".join([doc.page_content for doc in respuesta_rag['context']])
    
    # El juez evalúa la fidelidad
    evaluacion = evaluar_fidelidad(caso['pregunta'], contexto_usado, respuesta_bot)
    
    # Guardamos los datos
    resultados_finales.append({
        "ID": caso['id'],
        "Pregunta": caso['pregunta'],
        "Respuesta Bot": respuesta_bot,
        "Fidelidad (/10)": evaluacion.get('puntaje', 0), # Usamos .get() por seguridad
        "Motivo": evaluacion.get('razon', 'Sin motivo')
    })

# Convertimos a tabla (DataFrame) para visualizarlo bonito
df_reporte = pd.DataFrame(resultados_finales)

print("\n✅ Auditoría finalizada. Mostrando reporte:")
# Mostramos la tabla en Jupyter
display(df_reporte)

🚀 Iniciando auditoría automática...

Consultando: 'Compré el Cyberpunk 2077 hace 5 días y jugué 1 hora. ¿Puedo pedir reembolso?'...
Consultando: '¿Cuánta RAM necesito para jugar Lethal Company?'...
Consultando: 'Mi cuenta fue bloqueada, ¿por qué pudo haber sido?'...

✅ Auditoría finalizada. Mostrando reporte:


,ID,Pregunta,Respuesta Bot,Fidelidad (/10),Motivo
0,1,Compré el Cyberpunk 2077 hace 5 días y jugué 1...,"Sí, puedes pedir un reembolso. Según las polít...",10,La respuesta del bot es completamente fiel al ...
1,2,¿Cuánta RAM necesito para jugar Lethal Company?,No tengo información sobre eso.,10,El bot correctamente indicó que no tiene infor...
2,3,"Mi cuenta fue bloqueada, ¿por qué pudo haber s...",Tu cuenta pudo haber sido bloqueada por alguna...,10,La respuesta del bot se basa exclusivamente en...


### 📊 Análisis de Resultados (Conclusión)

Como se puede observar en la tabla superior, el sistema RAG ha sido evaluado exitosamente:
1. **Casos Válidos (ID 1 y 3):** El bot extrae correctamente la información de la base de datos vectorial y formula una respuesta útil.
2. **Mitigación de Alucinaciones (ID 2):** Ante una pregunta sobre un juego que no existe en nuestra base de datos (Lethal Company), el bot **no inventa requisitos**. Es honesto y deriva el problema, obteniendo un puntaje de Fidelidad de 10/10.

✅ **Veredicto:** El sistema es confiable, fiel a la documentación oficial de Steam, y está listo para integrar métricas adicionales (como Relevancia y Precisión del Contexto) en futuras iteraciones.